In [1]:
# after having them processed in medications_new.ipynb, format/create different csv's here:

In [2]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import os
%matplotlib widget
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':8})
pd.options.display.max_columns = 200
pd.options.display.max_rows = 300
pd.options.display.max_seq_items = 2000
# from tqdm import tqdm
from datetime import timedelta
import re
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file, get_metadata, read_in_routine

from medications import *

In [3]:
cohort_path = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/COV19_Sedation_Cohort.csv'
cohort = pd.read_csv(cohort_path)

In [4]:
cohort.PatientID.values;

In [5]:
cohort.head(2)

,PatientID,AdmissionDTS,DischargeDTS,OnsetDTS,Hospital,PatientEncounterID,MRN,BirthDTS,SexDSC,EthnicGroupDSC,PatientRaceDSC,Extubation,Intubation,CMO,DNR,DNI,Age,ICU_Admission,ICU_Discharge,ICU_LOS,Ventilation_On_DTS,Ventilation_Off_DTS,Ventilation_Length,PaO2_Arterial_ICU,PaO2_Arterial_ICU_DTS,PaO2_Arterial_PreVent,PaO2_Arterial_PreVent_DTS,PaO2_Arterial_PostVent,PaO2_Arterial_PostVent_DTS,PaO2_Venous_ICU,PaO2_Venous_ICU_DTS,PaO2_Venous_PreVent,PaO2_Venous_PreVent_DTS,PaO2_Venous_PostVent,PaO2_Venous_PostVent_DTS,ETT_On_DTS,ETT_Off_DTS,ETT_Length,PaO2_Arterial_PreETT,PaO2_Arterial_PreETT_DTS,PaO2_Arterial_PostETT,PaO2_Arterial_PostETT_DTS,PaO2_Venous_PreETT,PaO2_Venous_PreETT_DTS,PaO2_Venous_PostETT,PaO2_Venous_PostETT_DTS
0,Z8309928,2020-04-18 09:25:00,2020-06-22 18:13:00,2020-04-18 00:00:00,MGH,3303781737,10058176230,1938-05-14 00:00:00,Male,Not Hispanic or Latino,Black or African American,Yes,Yes,No,No,No,81,2020-04-22 02:36:00,2020-05-13 14:34:00,21.5,2020-04-22 03:47:00,2020-05-13 14:00:00,21.4,64.0,2020-04-22 06:00:00,NaN,NaN,64.0,2020-04-22 06:00:00,<28,2020-04-27 22:44:00,NaN,NaN,<28,2020-04-27 22:44:00,2020-04-22 03:40:00,2020-05-11 10:00:00,19.3,NaN,NaN,64.0,2020-04-22 06:00:00,NaN,NaN,<28,2020-04-27 22:44:00
1,Z10002895,2020-05-04 15:09:00,2020-05-19 11:38:00,2020-04-28 00:00:00,MGH,3305390010,10087535802,1963-12-13 00:00:00,Male,Not Hispanic or Latino,Other,Yes,Yes,No,No,No,56,2020-05-06 17:23:00,2020-05-15 13:23:00,8.8,2020-05-06 20:00:00,2020-05-14 15:00:00,7.8,223.0,2020-05-06 22:30:00,NaN,NaN,223.0,2020-05-06 22:30:00,NaN,NaN,NaN,NaN,NaN,NaN,2020-05-06 18:05:00,2020-05-15 03:00:00,8.4,NaN,NaN,223.0,2020-05-06 22:30:00,NaN,NaN,NaN,NaN


In [6]:
meds_all_output_dir_hourly_h5 = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/medications_all_timeseries_hourly_h5'
meds_all_output_dir_hourly_csv = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/medications_all_timeseries_hourly_csv'
meds_all_output_dir_hourly_wroute_csv = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/medications_all_timeseries_hourly_w_route_csv'

meds_spreadsheets_dir = '/media/mad3/Projects/Wolfgang/Covid19_Sedation/medications_spreadsheets'
if not os.path.exists(meds_spreadsheets_dir):
    os.makedirs(meds_spreadsheets_dir)


files = os.listdir(meds_all_output_dir_hourly_h5)
files[:3]
patient_ids = [file.split('_')[0] for file in files]

In [7]:
opioids = ['Oxycodone', 'Morphine', 'Fentanyl', 'Methadone', 'Hydromorphone',
       'Tramadol', 'Meperidine']
opioids = [x.lower() for x in opioids]

benzos = ['Lorazepam', 'Propofol', 'Midazolam', 'Clonazepam',
       'Dexmedetomidine', 'Alprazolam', 'Diazepam', 'Chlordiazepoxide']   
benzos = [x.lower() for x in benzos]
         
# meds_to_extract = opioids
# spreadsheet_name = 'opioids'


In [8]:
for meds_to_extract, spreadsheet_name in zip([
    opioids, benzos, ['opioids_sum'], ['benzos_sum'], ['gcs'], ['gcs_motor']],
['Opioids', 'Benzos', 'Opioids_Sum', 'Benzos_Sum', 'GCS', 'GCS_Motor']):

    df_extraction = pd.DataFrame()

    for file in files:
        
        patient_id = file.split('_')[0]
        cohort_sel = cohort.loc[cohort.PatientID == patient_id]

        if pd.notna(cohort_sel.ETT_On_DTS.values[0]):
            start_time = cohort_sel.ETT_On_DTS.values[0]
        elif pd.notna(cohort_sel.Ventilation_On_DTS.values[0]):
            start_time = cohort_sel.Ventilation_On_DTS.values[0]
        else:
            print(f'Both ETT and VentilationOn are missing, continue: {patient_id}')
        start_time = pd.Timestamp(start_time)

        if pd.notna(cohort_sel.DischargeDTS.values[0]):
            end_time = cohort_sel.DischargeDTS.values[0]
        else:
            print(f'Missing DischargeDTS: {patient_id}. Use large enough end date.')
            end_time = '2020-09-01'
            continue
        end_time = pd.Timestamp(end_time)


        data, hdr, fs = read_in_routine(os.path.join(meds_all_output_dir_hourly_h5, file))

        data = data.loc[start_time : end_time]

        meds_avail = [x for x in meds_to_extract if x in data.columns]
        data_patient = data[meds_avail].reset_index(drop=True)

        df_extraction_patient = pd.DataFrame(columns = meds_to_extract) # empty df with all columns of need
        data_patient = pd.concat([df_extraction_patient, data_patient], axis=0, sort=False)

        multiindex = pd.MultiIndex.from_product([[patient_id], meds_to_extract] , names=['PatientID', 'Medication'])
        data_patient = pd.DataFrame(data_patient.values, columns = multiindex)
        df_extraction = pd.concat([df_extraction, data_patient], axis=1)

    df_extraction.to_csv(os.path.join(meds_spreadsheets_dir, spreadsheet_name + '.csv'))

Missing DischargeDTS: Z12178374. Use large enough end date.
Missing DischargeDTS: Z10153042. Use large enough end date.
Both ETT and VentilationOn are missing, continue: Z10918133
Missing DischargeDTS: Z17768076. Use large enough end date.
Missing DischargeDTS: Z14032855. Use large enough end date.
Missing DischargeDTS: Z17800165. Use large enough end date.
Both ETT and VentilationOn are missing, continue: Z9142546
Missing DischargeDTS: Z14348226. Use large enough end date.
Missing DischargeDTS: Z9291915. Use large enough end date.
Missing DischargeDTS: Z10427477. Use large enough end date.
Missing DischargeDTS: Z17798960. Use large enough end date.
Both ETT and VentilationOn are missing, continue: Z11845750
Missing DischargeDTS: Z17798340. Use large enough end date.
Missing DischargeDTS: Z17809755. Use large enough end date.
Missing DischargeDTS: Z17761050. Use large enough end date.
Both ETT and VentilationOn are missing, continue: Z12828074
Missing DischargeDTS: Z17790585. Use large

In [11]:
df_extraction

PatientID,Z10916929,Z17759215,Z11879575,Z17756623,Z8839602,Z13248406,Z12850113,Z9913137,Z17761576,Z17790782,Z10095589,Z6925736,Z9103209,Z10082075,Z7122452,Z11429518,Z10609296,Z10531435,Z17082467,Z17769243,Z17752047,Z6940574,Z6435907,Z7525965,Z9444213,Z7335661,Z11102716,Z6922426,Z10912502,Z9334020,Z12815278,Z17770665,Z16981405,Z17760519,Z10977387,Z9914166,Z15009311,Z17756345,Z10420370,Z15598401,Z10912377,Z8635892,Z17760050,Z11177522,Z7825622,Z17761035,Z17764145,Z8058514,Z17760110,Z10251789,Z8264900,Z9844336,Z12652645,Z16993809,Z10579529,Z11674784,Z10918133,Z9371693,Z9308459,Z10369786,Z6732769,Z11218035,Z13755979,Z9428995,Z12002271,Z12137792,Z17753615,Z6566500,Z6544430,Z8930489,Z12017311,Z13735253,Z13777724,Z6702430,Z15865844,Z9329693,Z13851539,Z12263209,Z8309928,Z9085642,Z17760425,Z6752896,Z7657690,Z11358031,Z11907866,Z16486908,Z6729887,Z9149406,Z17753726,Z15410985,Z6730517,Z15651906,Z11595750,Z9033379,Z17758119,Z8326592,Z9889552,Z10259362,Z9134273,Z10454399,...,Z16628729,Z11752703,Z7316184,Z13731040,Z11346040,Z11359603,Z13569145,Z17778677,Z10235772,Z10946375,Z11705909,Z11170593,Z12823537,Z17785699,Z7022993,Z10331987,Z6854939,Z10131867,Z11630511,Z6456618,Z11608539,Z11231873,Z10152053,Z6825696,Z11287345,Z17767977,Z7167000,Z10303589,Z17750199,Z17760103,Z17773818,Z17761827,Z17160821,Z11842249,Z17755549,Z17755153,Z9089274,Z10876539,Z11405066,Z7217542,Z13887246,Z15927754,Z10260135,Z10292841,Z9966356,Z9154885,Z9700052,Z11845750,Z8805975,Z14341766,Z11410412,Z17753039,Z17755379,Z9250996,Z10926484,Z9128782,Z15780283,Z10714211,Z7911175,Z17748779,Z6844218,Z7934361,Z17758600,Z16014871,Z12390847,Z10766266,Z15075899,Z12828074,Z9615177,Z6933510,Z16591520,Z11352761,Z17777576,Z17746683,Z11392363,Z17723300,Z8567801,Z12714884,Z17787061,Z17756625,Z12775554,Z12156176,Z17756573,Z7054841,Z17772107,Z17748721,Z17084762,Z12076630,Z13765313,Z12962346,Z16209531,Z12308683,Z15089871,Z16525047,Z6545078,Z10767883,Z14210597,Z11708344,Z11654213,Z10063961
Medication,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,...,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor,gcs_motor
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [10]:
data.columns

Index(['gcs', 'gcs_eye', 'gcs_motor', 'gcs_verbal', 'lorazepam',
       'lorazepam_oral', 'lorazepam_parenteral', 'lorazepam_transdermal',
       'morphine', 'morphine_oral', 'morphine_parenteral',
       'morphine_transdermal', 'propofol', 'propofol_oral',
       'propofol_parenteral', 'propofol_transdermal', 'antipsychotics_sum',
       'benzos_sum', 'opioids_sum'],
      dtype='object')